In [ ]:
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path
x
def string_to_xml_file(xml_string, file_name):
    """
    Converts a string into a well-formatted (indented) XML file, without unnecessary newlines.

    Parameters:
    xml_string (str): The XML content as a string.
    file_name (str): The desired filename for the XML file.
    """
    try:
        # Parse the XML string
        root = ET.ElementTree(ET.fromstring(xml_string))
        
        # Convert ElementTree to a string
        rough_string = ET.tostring(root.getroot(), encoding="utf-8")
        
        # Use minidom to pretty-print the XML
        parsed = minidom.parseString(rough_string)
        pretty_xml_as_string = parsed.toprettyxml(indent="  ")
        
        # Remove unnecessary blank lines created by toprettyxml()
        pretty_xml_as_string = "\n".join([line for line in pretty_xml_as_string.splitlines() if line.strip()])
        
        # Write the formatted XML to a file
        with open(file_name, "w", encoding="utf-8") as f:
            f.write(pretty_xml_as_string)
        
        print(f"XML file '{file_name}' created successfully with proper indentation and no extra newlines.")
    except ET.ParseError as e:
        print("Error parsing XML string:", e)

# Source

## Data Sources
- Ministry of Agriculture, Food and Rural Affairs (MAFRA), 2021, 2050 Carbon Neutrality Strategy for the Agriculture and Food Sector. (`../resources/CNSAF-2021-MAFRA`)

## Implemented Input Files
- `/input/gcamdata/xml/N_Fert_reduction.xml`

* 2030 감축률: 2008 / 6334
* 2035 감축률: 2144 / 6569

In [1]:
2008 / 6334

0.31701926113040735

In [2]:
2144 / 6569

0.32638148881108237

In [7]:
from __future__ import annotations

from pathlib import Path
from decimal import Decimal, InvalidOperation
import xml.etree.ElementTree as ET
from collections import OrderedDict


# -------------------------
# User settings
# -------------------------
TARGET_TECHS = [
    'CornC4_Korea_RFD_hi', 'CornC4_Korea_RFD_lo',
    'FiberCrop_Korea_IRR_hi', 'FiberCrop_Korea_IRR_lo',
    'FiberCrop_Korea_RFD_hi', 'FiberCrop_Korea_RFD_lo',
    'FodderGrass_Korea_RFD_hi', 'FodderGrass_Korea_RFD_lo',
    'FruitsTree_Korea_IRR_hi', 'FruitsTree_Korea_IRR_lo',
    'FruitsTree_Korea_RFD_hi', 'FruitsTree_Korea_RFD_lo',
    'Fruits_Korea_IRR_hi', 'Fruits_Korea_IRR_lo',
    'Fruits_Korea_RFD_hi', 'Fruits_Korea_RFD_lo',
    'Legumes_Korea_RFD_hi', 'Legumes_Korea_RFD_lo',
    'MiscCrop_Korea_IRR_hi', 'MiscCrop_Korea_IRR_lo',
    'MiscCrop_Korea_RFD_hi', 'MiscCrop_Korea_RFD_lo',
    'NutsSeedsTree_Korea_IRR_hi', 'NutsSeedsTree_Korea_IRR_lo',
    'NutsSeedsTree_Korea_RFD_hi', 'NutsSeedsTree_Korea_RFD_lo',
    'NutsSeeds_Korea_RFD_hi', 'NutsSeeds_Korea_RFD_lo',
    'OilCrop_Korea_IRR_hi', 'OilCrop_Korea_IRR_lo',
    'OilCrop_Korea_RFD_hi', 'OilCrop_Korea_RFD_lo',
    'OtherGrainC4_Korea_RFD_hi', 'OtherGrainC4_Korea_RFD_lo',
    'OtherGrain_Korea_IRR_hi', 'OtherGrain_Korea_IRR_lo',
    'OtherGrain_Korea_RFD_hi', 'OtherGrain_Korea_RFD_lo',
    'RootTuber_Korea_IRR_hi', 'RootTuber_Korea_IRR_lo',
    'RootTuber_Korea_RFD_hi', 'RootTuber_Korea_RFD_lo',
    'Soybean_Korea_IRR_hi', 'Soybean_Korea_IRR_lo',
    'Soybean_Korea_RFD_hi', 'Soybean_Korea_RFD_lo',
    'Vegetables_Korea_IRR_hi', 'Vegetables_Korea_IRR_lo',
    'Vegetables_Korea_RFD_hi', 'Vegetables_Korea_RFD_lo',
    'Wheat_Korea_RFD_hi', 'Wheat_Korea_RFD_lo'
]

# reduction factors: new = old * factor
YEAR_FACTOR = {
    2030: Decimal("1") - Decimal("0.317"),  # 0.683
    2035: Decimal("1") - Decimal("0.326"),  # 0.674
}

GAS_NAME = "N2O_AWB"


# -------------------------
# Helpers
# -------------------------
def _to_decimal(s: str) -> Decimal:
    try:
        return Decimal(s.strip())
    except (InvalidOperation, AttributeError):
        raise ValueError(f"Cannot parse Decimal from: {s!r}")


def _indent(elem: ET.Element, level: int = 0) -> None:
    """Pretty-print indentation (in-place)."""
    i = "\n" + level * "  "
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + "  "
        for child in elem:
            _indent(child, level + 1)
        if not elem.tail or not elem.tail.strip():
            elem.tail = i
    else:
        if level and (not elem.tail or not elem.tail.strip()):
            elem.tail = i


def _iter_tech_blocks(root: ET.Element):
    """
    Traverse AgSupplySector -> AgSupplySubsector -> AgProductionTechnology.

    Yields: (sector_name, subsector_name, tech_elem)
    """
    for region in root.iter("region"):
        if region.get("name") != "South Korea":
            continue
        for sector in region.iter("AgSupplySector"):
            sec_name = sector.get("name", "")
            for subsector in sector.iter("AgSupplySubsector"):
                sub_name = subsector.get("name", "")
                for tech in subsector.iter("AgProductionTechnology"):
                    yield sec_name, sub_name, tech


def _extract_year_to_emisscoef(tech: ET.Element) -> dict[int, Decimal]:
    """
    Extract emiss-coef for GAS_NAME at years 2030/2035 from a tech element.

    Supports both:
      (A) tech contains <period year="2030"> ... <Non-CO2 ...>
      (B) tech itself is year-specific: <AgProductionTechnology year="2030" ...> ... <Non-CO2 ...>
          (in that case, we treat it as year y without needing <period>)
    Returns: {year: coeff_decimal} for found years only.
    """
    out: dict[int, Decimal] = {}

    # Case A: has explicit periods
    periods = tech.findall(".//period")
    if periods:
        for p in periods:
            y_str = p.get("year")
            if y_str is None:
                continue
            try:
                y = int(y_str)
            except ValueError:
                continue
            if y not in YEAR_FACTOR:
                continue

            # find Non-CO2 with name=GAS_NAME inside this period
            for nc in p.iter():
                if nc.tag in ("Non-CO2", "NonCO2") and nc.get("name") == GAS_NAME:
                    ec = nc.find("emiss-coef")
                    if ec is None or ec.text is None:
                        continue
                    out[y] = _to_decimal(ec.text)
        return out

    # Case B: tech is year-specific via attribute
    y_str = tech.get("year")
    if y_str is None:
        return out
    try:
        y = int(y_str)
    except ValueError:
        return out
    if y not in YEAR_FACTOR:
        return out

    for nc in tech.iter():
        if nc.tag in ("Non-CO2", "NonCO2") and nc.get("name") == GAS_NAME:
            ec = nc.find("emiss-coef")
            if ec is None or ec.text is None:
                continue
            out[y] = _to_decimal(ec.text)
    return out


# -------------------------
# Main: build minimal policy xml
# -------------------------
def build_n_fert_reduction_policy_xml(
    in_xml: str,
    out_xml: str,
    target_techs: list[str] = TARGET_TECHS,
) -> None:
    target_set = set(target_techs)

    src_tree = ET.parse(in_xml)
    src_root = src_tree.getroot()

    # Collect in an ordered structure to preserve "first-seen" ordering
    # sectors[sec][sub][tech] -> dict(year->Decimal emisscoef)
    sectors: "OrderedDict[str, OrderedDict[str, OrderedDict[str, dict[int, Decimal]]]]" = OrderedDict()

    checked_nodes = 0

    for sec_name, sub_name, tech in _iter_tech_blocks(src_root):
        tech_name = tech.get("name", "")
        if tech_name not in target_set:
            continue

        year_to_coef = _extract_year_to_emisscoef(tech)
        if not year_to_coef:
            continue

        checked_nodes += len(year_to_coef)

        sectors.setdefault(sec_name, OrderedDict())
        sectors[sec_name].setdefault(sub_name, OrderedDict())
        sectors[sec_name][sub_name].setdefault(tech_name, {})

        # Merge (in case same tech appears multiple times; keep last encountered per year)
        for y, coef in year_to_coef.items():
            sectors[sec_name][sub_name][tech_name][y] = coef

    # ---- Build new XML from scratch (minimal policy style) ----
    scenario = ET.Element("scenario")
    world = ET.SubElement(scenario, "world")
    region = ET.SubElement(world, "region", {"name": "South Korea"})

    modified = 0
    missing_years = 0

    for sec_name, sub_dict in sectors.items():
        sec_el = ET.SubElement(region, "AgSupplySector", {"name": sec_name, "nocreate": "1"})
        for sub_name, tech_dict in sub_dict.items():
            sub_el = ET.SubElement(sec_el, "AgSupplySubsector", {"name": sub_name, "nocreate": "1"})
            for tech_name, ycoef in tech_dict.items():
                tech_el = ET.SubElement(sub_el, "AgProductionTechnology", {"name": tech_name, "nocreate": "1"})

                # Always create periods in output (2030/2035), only if we have source value
                for y in (2030, 2035):
                    if y not in ycoef:
                        missing_years += 1
                        continue

                    factor = YEAR_FACTOR[y]
                    new_coef = ycoef[y] * factor

                    per = ET.SubElement(tech_el, "period", {"year": str(y)})
                    nc = ET.SubElement(per, "Non-CO2", {"name": GAS_NAME})
                    ec = ET.SubElement(nc, "emiss-coef")
                    ec.text = f"{new_coef:.8e}"  # scientific notation, stable formatting

                    modified += 1

    _indent(scenario)

    out_path = Path(out_xml)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    ET.ElementTree(scenario).write(str(out_path), encoding="UTF-8", xml_declaration=True)

    print(f"Done. Source year-coef pairs found: {checked_nodes}")
    print(f"Output periods written (modified): {modified}")
    if missing_years:
        print(f"Note. Missing requested years (2030/2035) for some techs: {missing_years}")
    print(f"Output written to: {out_xml}")


In [8]:
build_n_fert_reduction_policy_xml(
    in_xml="../../exe/debugBuildings-CP.xml",
    out_xml="../../input/policy/korea-2035/agriculture/N_fert_reduction.xml",
)

Done. Source year-coef pairs found: 104
Output periods written (modified): 104
Output written to: ../../input/policy/korea-2035/agriculture/N_fert_reduction.xml


# Nitrogen Fertilizer Reduction Policy for Cropland

South Korea plans to reduce nitrogen fertilizer application rates from **262 kg/ha** (2019 baseline) to **115 kg/ha**, representing a **43.9% reduction**.  

GWP reduction rates are applied to the default GCAM emissions coefficients to ensure model consistency.  
Below are calculations for implementing the policy.

In [30]:
def modify_fertilizer_coefficient(input_file, output_file):
    """
    Reduces the N fertilizer coefficient by 56.1% for all agricultural
    production technologies in South Korea from the year 2030 onwards.

    Args:
        input_file (str): The path to the input XML file.
        output_file (str): The path to the output XML file.
    """
    tree = ET.parse(input_file)
    root = tree.getroot()

    # Find the region "South Korea"
    for region in root.findall('.//region[@name="South Korea"]'):
        # Iterate through all AgProductionTechnology elements
        for tech in region.findall('.//AgProductionTechnology'):
            # Find all period elements from 2030 onwards
            for period in tech.findall('period'):
                year = int(period.get('year'))
                if year >= 2030:
                    # Find the N fertilizer input
                    for n_fertilizer in period.findall('.//minicam-energy-input[@name="N fertilizer"]'):
                        # Get the coefficient and reduce it
                        coefficient_element = n_fertilizer.find('coefficient')
                        if coefficient_element is not None:
                            original_coefficient = float(coefficient_element.text)
                            new_coefficient = original_coefficient * 0.561
                            coefficient_element.text = str(new_coefficient)

    xml_string = ET.tostring(root, encoding="unicode")
    string_to_xml_file(xml_string, output_file)

In [ ]:
input_file = "../../input/gcamdata/xml/ag_Fert_IRR_MGMT.xml"
output_file = "../../input/policy/korea-2035/agriculture/N_Fert_reduction.xml"

In [32]:
modify_fertilizer_coefficient(input_file, output_file)

XML file '/home/hyuntae-choi/gcam-core/input/policy/korea-2035/agriculture/N_Fert_reduction.xml' created successfully with proper indentation and no extra newlines.
